In [0]:
# CAPA REFINED — aqui le damos valor a los datos limpios
# tres KPIs que responden preguntas reales del negocio


import time
from pyspark.sql import functions as f

CATALOGO     = "nyc_taxi_andres"
CAPA_RAW     = f"{CATALOGO}.raw"
CAPA_TRUSTED = f"{CATALOGO}.trusted"
CAPA_REFINED = f"{CATALOGO}.refined"

print("Cargando datos desde Trusted...")
inicio = time.time()

viajes = spark.table(f"{CAPA_TRUSTED}.viajes_limpios")

total = viajes.count()
print(f"Viajes disponibles para los KPIs: {total:,}")

In [0]:
# definimos las 6 franjas horarias del dia
# estas franjas tienen sentido para el negocio de taxis en NYC

viajes = viajes.withColumn(
    "hora", f.hour("fecha_hora_inicio")
).withColumn(
    "dia_semana", f.date_format("fecha_hora_inicio", "EEEE")
).withColumn(
    "franja_horaria",
    f.when(f.col("hora").between(0, 5),  "1_madrugada")
     .when(f.col("hora").between(6, 8),  "2_manana_temprano")
     .when(f.col("hora").between(9, 11), "3_manana")
     .when(f.col("hora").between(12, 16),"4_tarde")
     .when(f.col("hora").between(17, 19),"5_hora_pico")
     .otherwise("6_noche")
)

print("Franjas horarias creadas")
print("Distribucion por franja:")
viajes.groupBy("franja_horaria").count().orderBy("franja_horaria").show()

In [0]:
# KPI 1: cuantos viajes hay, cuanto duran y cuanto cobran
# agrupado por franja horaria y dia de la semana

print("Calculando KPI 1 - patron de demanda temporal...")
inicio = time.time()

kpi_demanda = viajes.groupBy("franja_horaria", "dia_semana").agg(
    f.count("*").alias("cantidad_viajes"),
    f.round(f.avg("duracion_horas") * 60, 2).alias("duracion_promedio_minutos"),
    f.round(f.avg("tarifa_base"), 2).alias("tarifa_promedio"),
    f.round(f.avg("total_cobrado"), 2).alias("total_promedio")
).orderBy("franja_horaria", "dia_semana")

kpi_demanda.write.format("delta").mode("overwrite").saveAsTable(
    f"{CAPA_REFINED}.kpi_demanda_temporal"
)

print(f"KPI 1 guardado en {round(time.time() - inicio, 2)}s")
print(f"Registros generados: {kpi_demanda.count()}")

# identificamos los picos — las franjas con mas viajes
print("\nTop 5 combinaciones franja/dia con mas viajes:")
kpi_demanda.orderBy(f.desc("cantidad_viajes")).show(5)

In [0]:
# KPI 2: que zonas son mas rentables para un taxista?
# medimos ingreso promedio por viaje y velocidad promedio
# para la velocidad filtramos viajes menores a 1 minuto para evitar distorsion
# no los descartamos del dataset porque son transacciones reales

print("Calculando KPI 2 - eficiencia economica por zona...")
inicio = time.time()

# filtramos viajes menores a 1 minuto solo para este calculo
viajes_con_velocidad = viajes.filter(f.col("duracion_horas") >= (1/60))

kpi_zonas = viajes_con_velocidad.groupBy("barrio_origen", "zona_origen").agg(
    f.count("*").alias("cantidad_viajes"),
    f.round(f.avg(f.col("total_cobrado") / f.col("distancia_millas")), 2).alias("ingreso_por_milla"),
    f.round(f.avg(f.col("distancia_millas") / f.col("duracion_horas")), 2).alias("velocidad_promedio_mph"),
    f.round(f.avg("total_cobrado"), 2).alias("ingreso_promedio"),
    f.round(f.sum("total_cobrado"), 2).alias("ingreso_total")
).filter(
    # excluimos zonas sin identificar porque no representan un lugar real de NYC
    # minimo 50 viajes para que la zona sea estadisticamente representativa
    (f.col("barrio_origen") != "N/A") &
    (f.col("barrio_origen") != "Unknown") &
    (f.col("cantidad_viajes") >= 50)
).orderBy(f.desc("ingreso_promedio"))

top_10_zonas = kpi_zonas.limit(10)

top_10_zonas.write.format("delta").mode("overwrite").saveAsTable(
    f"{CAPA_REFINED}.kpi_top10_zonas_rentables"
)

print(f"KPI 2 guardado en {round(time.time() - inicio, 2)}s")
print("\nTop 10 zonas mas rentables por ingreso promedio por viaje:")
top_10_zonas.show(10, truncate=False)

In [0]:
# KPI 3: cuanta plata representan los registros que descartamos?
# comparamos los ingresos totales con y sin los filtros aplicados


# ingresos con los datos limpios
ingresos_limpios = viajes.agg(
    f.round(f.sum("total_cobrado"), 2).alias("total")
).collect()[0]["total"]

# leemos los datos crudos para comparar
viajes_crudos = spark.table(f"{CAPA_RAW}.viajes_enero_2023")
ingresos_crudos = viajes_crudos.agg(
    f.round(f.sum("total_amount"), 2).alias("total")
).collect()[0]["total"]

diferencia = round(ingresos_crudos - ingresos_limpios, 2)
pct_impacto = round((diferencia / ingresos_crudos) * 100, 2)

# definimos los descartados manualmente con los numeros que ya tenemos
descartados = {
    "tiempo_invalido": 1121,
    "sin_distancia": 44814,
    "tarifa_invalida": 22522,
    "outliers_extremos": 3046,
    "nulos_columnas_clave": 0
}
total_descartados = sum(descartados.values())
total_inicial = 3066766

# construimos la tabla del reporte de calidad
from pyspark.sql import Row

filas = [
    Row(
        regla=regla,
        registros_descartados=int(cantidad),
        pct_del_total=round((cantidad / total_inicial) * 100, 2)
    )
    for regla, cantidad in descartados.items()
]

reporte_calidad = spark.createDataFrame(filas).withColumn(
    "ingresos_totales_con_filtros", f.lit(ingresos_limpios)
).withColumn(
    "ingresos_totales_sin_filtros", f.lit(ingresos_crudos)
).withColumn(
    "diferencia_ingresos", f.lit(diferencia)
).withColumn(
    "pct_impacto_ingresos", f.lit(pct_impacto)
)

reporte_calidad.write.format("delta").mode("overwrite").saveAsTable(
    f"{CAPA_REFINED}.kpi_calidad_datos"
)

print(f"Ingresos con datos limpios : ${ingresos_limpios:,.2f}")
print(f"Ingresos con datos crudos  : ${ingresos_crudos:,.2f}")
print(f"Diferencia                 : ${diferencia:,.2f} ({pct_impacto}%)")
print("KPI 3 guardado")

In [0]:
print("  RESUMEN — CAPA REFINED")
print("=" * 55)
print(f"  KPI 1 - demanda temporal  : guardado")
print(f"  KPI 2 - top 10 zonas      : guardado")
print(f"  KPI 3 - impacto calidad   : guardado")
print("")
print(f"  Tablas en refined:")
print(f"    - kpi_demanda_temporal")
print(f"    - kpi_top10_zonas_rentables")
print(f"    - kpi_calidad_datos")
print("=" * 55)
print("  Pipeline Medallion completo")
